In [ ]:
# Fix: Kaggle's default NumPy 1.26.4 is CORRECT - prevent upgrades to 2.x
# Problem: Ultralytics pulls in NumPy 2.x, but Kaggle's OpenCV 4.12.0 was compiled with NumPy 1.x
# Solution: Install ultralytics, then LOCK NumPy at 1.26.4
import sys

print("Step 1: Record Kaggle's default NumPy version...")
import numpy as np

default_numpy = np.__version__
print(f"   Default NumPy: {default_numpy}")

print("\nStep 2: Install ultralytics...")
import subprocess

subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)

print("\nStep 3: LOCK NumPy back to Kaggle's default (prevent 2.x upgrade)...")
subprocess.run(
    ["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"],
    check=True,
)

print("\nInstallation complete!")
print("\nVerifying versions...")
import numpy as np

print(f"   NumPy: {np.__version__}")

from ultralytics import YOLO

print("   OK Ultralytics imported successfully")

In [ ]:
import json
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

# Environment detection
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# KAGGLE DATASET SETUP: Add code dataset directly to path
if os.path.exists("/kaggle/input/wound-seg-code-v1"):
    sys.path.insert(0, "/kaggle/input/wound-seg-code-v1")
    print("[INFO] Code dataset added to Python path")
    # On Kaggle, import without src. prefix
    from evaluation.metrics import (
        calculate_bootstrap_ci,
        calculate_boundary_f1,
        calculate_segmentation_metrics,
    )
    from pipeline.eda import EDA_CONFIG, Visualizer
else:
    # Local development: add project root to path
    sys.path.insert(0, str(PROJECT_ROOT))
    print("[INFO] Using local src directory")
    # Locally, import with src. prefix
    from src.evaluation.metrics import (
        calculate_bootstrap_ci,
        calculate_boundary_f1,
        calculate_segmentation_metrics,
    )
    from src.pipeline.eda import EDA_CONFIG, Visualizer

print("OK All modules imported successfully")

In [ ]:
# Detect environment (Kaggle vs local)
IN_KAGGLE = Path("/kaggle/input").exists()

if IN_KAGGLE:
    # Kaggle dataset has extra 'data_wound_seg' parent directory
    BASE_INPUT_PATH = Path("/kaggle/input/baseline-outputs/data_wound_seg")
    OUTPUT_PATH = Path("/kaggle/working")
else:
    # Local development: use kaggle_outputs directory for this run
    BASE_INPUT_PATH = PROJECT_ROOT / "simulated_kaggle_data/input/wound-segmentation-images"
    OUTPUT_PATH = PROJECT_ROOT / "kaggle_outputs/ablation_3_counterfactual"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Local (simulated)'}")
print(f"Input path: {BASE_INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Project root: {PROJECT_ROOT}")

---

# Ablation 3 Counterfactual: Multi-Scale Training Without RandAugment

**Research Question**: Open: is multi-scale training beneficial when RandAugment is disabled.

**Hypothesis**: Original Ablation 3 failed from the RandAugment conflict, not from multi-scale. Disabling RandAugment shows the multi-scale effect.

**Implementation**:
- **Multi-scale**: `imgsz=[416, 512, 640]` (SAME as original)
- **RandAugment**: disabled with the `copy_paste=0.0` workaround
- **Mosaic**: DISABLED (`mosaic=0.0`) to isolate multi-scale effect
- **Other augmentations**: Kept at baseline defaults

**YOLOv8 Bug Workaround**:
- YOLOv8 ignores `auto_augment=None`
- Solution: `copy_paste=0.0` disables the RandAugment pipeline
- Source: ultralytics/issues/4710, Stack Overflow research

**Expected Outcome**: Dice ~78-82% (recovery from 4.8% failure)

---

In [ ]:
# Load pretrained model (automatically downloads yolov8n-seg.pt if not present)
print("Loading pretrained YOLOv8n-seg (COCO weights)...")
model = YOLO("yolov8n-seg.pt")

print(f"Model loaded successfully")
print(f"Model classes: {len(model.names)} COCO classes")
print(f"Sample classes: {list(model.names.values())[:5]}")

In [ ]:
# Paths to test data
TEST_IMAGES_DIR = BASE_INPUT_PATH / "test/images"
TEST_MASKS_DIR = BASE_INPUT_PATH / "test_masks"  # PNG masks for evaluation

# Get all test images (files are directly in test_images/)
test_image_paths = sorted(TEST_IMAGES_DIR.glob("*.png")) if TEST_IMAGES_DIR.exists() else []
print(f"Found {len(test_image_paths)} test images")

# Verify corresponding masks exist and store as string paths
test_pairs = []
for img_path in test_image_paths:
    mask_path = TEST_MASKS_DIR / img_path.name
    if mask_path.exists():
        # Convert to strings immediately to avoid Path object issues
        test_pairs.append((str(img_path), str(mask_path)))
    else:
        print(f"WARNING: Missing mask for {img_path.name}")

print(f"Valid test pairs: {len(test_pairs)}")

# Show first test pair
if test_pairs:
    print(f"\nFirst test pair:")
    print(f"  Image: {test_pairs[0][0]}")
    print(f"  Mask: {test_pairs[0][1]}")
elif len(test_image_paths) == 0:
    raise FileNotFoundError(
        f"No test images found in {TEST_IMAGES_DIR}. "
        f"Please check dataset structure in {BASE_INPUT_PATH}"
    )

In [ ]:
# Import additional modules for fine-tuning
if os.path.exists("/kaggle/input/wound-seg-code-v1"):
    # On Kaggle, import without src. prefix
    from data.yolo_config import create_yolo_config, validate_yolo_config
    from evaluation.yolo_utils import yolo_result_to_metrics_input
else:
    # Local development: import with src. prefix
    from src.data.yolo_config import create_yolo_config, validate_yolo_config
    from src.evaluation.yolo_utils import yolo_result_to_metrics_input

import shutil
import time

print("OK Fine-tuning modules imported successfully")

In [ ]:
# YOLO Label Conversion - SKIPPED (labels pre-converted in baseline-outputs!)
print("=" * 80)
print("USING PRE-CONVERTED YOLO LABELS FROM BASELINE-OUTPUTS")
print("=" * 80)

# baseline-outputs already has:
# - data_wound_seg/train/images/ (2,208 images)
# - data_wound_seg/train/labels/ (2,208 YOLO .txt files)
# - data_wound_seg/test/images/ (552 images)
# - data_wound_seg/test/labels/ (552 YOLO .txt files)

# Use dataset directly (no conversion needed!)
YOLO_DATASET_ROOT = BASE_INPUT_PATH

print(f"\nDataset root: {YOLO_DATASET_ROOT}")
print(f"  Train: {len(list((YOLO_DATASET_ROOT / 'train/images').glob('*.png')))} images")
print(f"  Train labels: {len(list((YOLO_DATASET_ROOT / 'train/labels').glob('*.txt')))} files")
print(f"  Test: {len(list((YOLO_DATASET_ROOT / 'test/images').glob('*.png')))} images")
print(f"  Test labels: {len(list((YOLO_DATASET_ROOT / 'test/labels').glob('*.txt')))} files")

print(f"\n" + "=" * 80)
print("YOLO DATASET READY (PRE-CONVERTED)")
print("=" * 80)

In [ ]:
# Generate YOLO configuration file (now that labels exist)
print(f"\nGenerating YOLO dataset configuration...")

config_result = create_yolo_config(
    dataset_root=YOLO_DATASET_ROOT,
    train_dir="train",  # Now points to train/ (with images/ and labels/)
    val_dir="test",  # Now points to test/ (with images/ and labels/)
    test_dir="test",
    output_path=OUTPUT_PATH / "wound_dataset.yaml",  # Write to /kaggle/working (writable)
)

if config_result["success"]:
    config_path = config_result["data"]["config_path"]
    print(f"OK Configuration created: {config_path}")

    # Validate the configuration
    validation_result = validate_yolo_config(config_path)
    if validation_result["success"]:
        print(f"OK Configuration validated successfully")
        print(f"  Dataset root: {validation_result['data']['path']}")
        print(f"  Number of classes: {validation_result['data']['nc']}")
        print(f"  Class names: {validation_result['data']['names']}")
        print(f"  Train path: {validation_result['data']['train']}")
        print(f"  Val path: {validation_result['data']['val']}")
    else:
        raise ValueError(f"Configuration validation failed: {validation_result['error']}")
else:
    raise ValueError(f"Configuration creation failed: {config_result['error']}")

print(f"\n" + "=" * 60)
print("Dataset prepared and validated for training WITH LABELS")
print("=" * 60)

---

## Training Configuration Comparison

| Parameter | Original Ablation 3 | This Counterfactual | Change |
|-----------|---------------------|---------------------|--------|
| `imgsz` | `[416, 512, 640]` | `[416, 512, 640]` | same |
| `mosaic` | `0.0` | `0.0` | same |
| `auto_augment` | `None` (ignored) | N/A | - |
| `copy_paste` | Default (0.1) | `0.0` | only change |
| **Effect** | RandAugment active | RandAugment disabled | **Isolates RandAugment** |

**Fix**: The original used `auto_augment=None` and YOLOv8 ignored it. This counterfactual uses `copy_paste=0.0` to disable the RandAugment pipeline.

**Expected**: Dice recovers to about 78-82% if multi-scale helps. If it still fails, multi-scale is the problem.

---

In [ ]:
# Multi-seed training with Multi-Scale + NO RandAugment (P100-Optimized)
RANDOM_SEEDS = [42, 123, 456]

# P100-Optimized Hyperparameters (SAME as baseline for fair comparison)
BATCH_SIZE = 32
LEARNING_RATE = 0.015
WORKERS = 8
AMP = False

# Dataset-Aligned Parameters
IMAGE_SIZE = [416, 512, 640]  # Multi-scale (SAME as original)
EPOCHS = 100
PATIENCE = 10

# Storage for all training results
all_training_results = {}

print("=" * 80)
print("ABLATION 3 COUNTERFACTUAL: MULTI-SCALE WITHOUT RANDAUGMENT")
print("=" * 80)
print(f"\nTraining configuration:")
print(f"  Seeds: {RANDOM_SEEDS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Multi-scale: [416, 512, 640] pixels (SAME as original)")
print(f"  Mosaic: DISABLED (0.0) - SAME as original")
print(f"  RandAugment: PROPERLY DISABLED (copy_paste=0.0)")
print(f"\nDataset config: {config_path}")
print(f"\nEstimated time: ~1.5-2 hours total (~30-40 min per seed on P100)\n")

for seed_idx, seed in enumerate(RANDOM_SEEDS, 1):
    print("\n" + "=" * 80)
    print(f"TRAINING WITH SEED {seed} ({seed_idx}/{len(RANDOM_SEEDS)})")
    print("=" * 80)

    # Check if weights already exist from previous run
    weights_path = OUTPUT_PATH / "yolo_training" / f"seed_{seed}" / "weights" / "best.pt"

    if weights_path.exists():
        print(f"OK Found existing weights: {weights_path}")
        print(f"  Loading pre-trained model (skipping training)...")
        model = YOLO(str(weights_path))
        training_skipped = True
    else:
        print(f"WARNING No existing weights found - training from scratch")
        print(f"  Expected path: {weights_path}")
        print(f"  This will take ~30-40 minutes on P100...")

        # Initialize model from pretrained COCO weights
        model = YOLO("yolov8n-seg.pt")
        training_skipped = False

        # Train model
        training_start = time.time()

        results = model.train(
            data=str(config_path),
            # P100-optimized parameters
            batch=BATCH_SIZE,
            lr0=LEARNING_RATE,
            workers=WORKERS,
            amp=AMP,
            # Dataset-aligned
            epochs=EPOCHS,
            patience=PATIENCE,
            # Multi-Scale Training (SAME as original)
            imgsz=IMAGE_SIZE,
            # Disable mosaic (SAME as original)
            mosaic=0.0,
            close_mosaic=0,
            # CRITICAL FIX: Properly disable RandAugment
            copy_paste=0.0,  # This disables RandAugment pipeline
            # Reproducibility
            seed=seed,
            save=True,
            project=str(OUTPUT_PATH / "yolo_training"),
            name=f"seed_{seed}",
            exist_ok=True,
            verbose=False,
        )

        training_duration = time.time() - training_start
        print(f"\nOK Training completed in {training_duration/60:.1f} minutes")

    # EVALUATION (always runs, whether weights were loaded or trained from scratch)
    print(f"\nEvaluating model on test set ({len(test_pairs)} images)...")

    seed_results = []
    for idx, (img_path_str, mask_path_str) in enumerate(test_pairs):
        # Extract image ID from path string
        image_id = Path(img_path_str).stem

        # Load image
        img = cv2.imread(img_path_str)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Load ground truth mask
        gt_mask = cv2.imread(mask_path_str, cv2.IMREAD_GRAYSCALE)
        _, gt_mask = cv2.threshold(gt_mask, 127, 1, cv2.THRESH_BINARY)

        # Run inference
        yolo_results = model.predict(img_rgb, verbose=False)

        # Convert YOLO result to metrics input format
        conversion_result = yolo_result_to_metrics_input(yolo_results[0], mask_path_str, image_id)

        if conversion_result["success"]:
            pred_mask = conversion_result["data"]["pred_mask"]
            gt_mask_from_result = conversion_result["data"]["gt_mask"]

            # Calculate segmentation metrics
            metrics = calculate_segmentation_metrics(pred_mask, gt_mask, image_id)

            # Calculate boundary F1
            boundary_f1 = calculate_boundary_f1(pred_mask, gt_mask, tolerance=2)
            metrics["boundary_f1"] = boundary_f1

            seed_results.append(metrics)
        else:
            print(f"  WARNING: Failed to process {image_id}: {conversion_result['error']}")

        # Progress update
        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx + 1}/{len(test_pairs)} images...")

    # Store results for this seed
    all_training_results[f"seed_{seed}"] = seed_results

    print(f"\nOK Seed {seed} complete: {len(seed_results)} images evaluated")

print("\n" + "=" * 80)
print("ALL TRAINING/EVALUATION RUNS COMPLETE")
print("=" * 80)
print(f"Total seeds: {len(all_training_results)}")
print(f"Images per seed: {len(next(iter(all_training_results.values())))}")

## Results Analysis

**Scenario A: Dice ~78-82%** (Recovery from 4.8% failure)
- **Interpretation**: the RandAugment conflict was the cause
- **Conclusion**: multi-scale training helps when RandAugment is disabled
- **Action**: document the RandAugment incompatibility with multi-scale

**Scenario B: Dice <60%** (Still fails)
- **Interpretation**: multi-scale is the problem for wound segmentation
- **Conclusion**: single-scale (512) fits this dataset best
- **Action**: drop multi-scale, use single-scale in later ablations

In [ ]:
# Aggregate results across all seeds
print("=" * 80)
print("AGGREGATING COUNTERFACTUAL RESULTS")
print("=" * 80)

# Pool all results from all seeds
all_counterfactual_results = []
for seed_key, seed_results in all_training_results.items():
    all_counterfactual_results.extend(seed_results)

print(f"\nTotal predictions pooled: {len(all_counterfactual_results)}")
print(f"Expected: {len(test_pairs) * len(RANDOM_SEEDS)}")

# Calculate aggregate statistics with bootstrap CIs
cf_dice_scores = [r["dice"] for r in all_counterfactual_results]
cf_iou_scores = [r["iou"] for r in all_counterfactual_results]
cf_precision_scores = [r["precision"] for r in all_counterfactual_results]
cf_recall_scores = [r["recall"] for r in all_counterfactual_results]
cf_boundary_f1_scores = [r["boundary_f1"] for r in all_counterfactual_results]

print(f"\nCalculating bootstrap confidence intervals...\n")

cf_summary_stats = {}

for metric_name, metric_values in [
    ("dice", cf_dice_scores),
    ("iou", cf_iou_scores),
    ("precision", cf_precision_scores),
    ("recall", cf_recall_scores),
    ("boundary_f1", cf_boundary_f1_scores),
]:
    ci_result = calculate_bootstrap_ci(
        metric_values, confidence_level=0.95, n_bootstrap=10000, random_seed=42
    )
    cf_summary_stats[metric_name] = ci_result

    # Compare to baseline
    baseline_mean = 0.8000
    improvement = ci_result["mean"] - baseline_mean

    print(f"{metric_name.upper()}:")
    print(f"  Counterfactual:  {ci_result['mean']:.4f} ± {ci_result['std']:.4f}")
    print(f"  95% CI: [{ci_result['ci_lower']:.4f}, {ci_result['ci_upper']:.4f}]")
    print(f"  Baseline:        {baseline_mean:.4f}")
    print(f"  Change:          {improvement:+.4f} ({100*improvement/baseline_mean:+.1f}%)")
    print()

print("=" * 80)
print("COUNTERFACTUAL RESULTS COMPLETE")
print("=" * 80)

In [ ]:
# Ablation 3 Counterfactual completion status
completion_status = {
    "notebook": "ablation_3_counterfactual_no_randaugment",
    "ablation": "multiscale_counterfactual",
    "status": "complete",
    "intervention": "Multi-scale [416,512,640] with RandAugment PROPERLY disabled",
    "workaround": "copy_paste=0.0 to disable RandAugment pipeline",
    "n_images_evaluated": len(test_pairs),
    "n_seeds": len(RANDOM_SEEDS),
    "seeds": RANDOM_SEEDS,
    "counterfactual_dice_mean": cf_summary_stats["dice"]["mean"],
    "baseline_dice_mean": 0.8000,
    "dice_change_vs_baseline": cf_summary_stats["dice"]["mean"] - 0.8000,
    "original_ablation_3_dice": 0.048,  # 4.8% catastrophic failure
    "recovery_from_failure": cf_summary_stats["dice"]["mean"] - 0.048,
    "environment": "Kaggle" if IN_KAGGLE else "Local",
}

completion_path = OUTPUT_PATH / "completion_status.json"
with open(completion_path, "w") as f:
    json.dump(completion_status, f, indent=2)

print(f"\nCompletion status saved: {completion_path}")

print("\n" + "=" * 80)
print("OK ABLATION 3 COUNTERFACTUAL COMPLETE")
print("=" * 80)
print(f"\nResults:")
print(f"  Dice Score:      {cf_summary_stats['dice']['mean']:.4f}")
print(f"  Baseline:        0.8000")
print(f"  Change:          {cf_summary_stats['dice']['mean'] - 0.8000:+.4f}")
print(f"  Original A3:     0.0480 (catastrophic failure)")
print(f"  Recovery:        {cf_summary_stats['dice']['mean'] - 0.048:+.4f}")
print(f"\nAll outputs saved to: {OUTPUT_PATH}")
print("=" * 80 + "\n")